## Project 4: Automated CSV Explorer
Write a script that accepts any CSV path as a
command-line argument, auto-detects numeric
columns, generates summary statistics using
Pandas, plots histograms for all numeric
columns, and saves a self-contained HTML
report embedding the plots as base64 images.
Skills: sys.argv, Pandas, Matplotlib, io.BytesIO,
base64, HTML generation

In [8]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import io
import base64

sys.argv = ['project_4.ipynb', 'System_Anomalies.csv']

def generate_report(csv_path):
    # 1. Load the Data
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    # 2. Auto-detect Numeric Columns
    numeric_df = df.select_dtypes(include=['number'])
    
    if numeric_df.empty:
        print("No numeric columns found to analyze.")
        return

    # 3. Generate Summary Statistics
    # We use to_html() to convert the dataframe directly into an HTML table
    stats_table = numeric_df.describe().to_html(classes='table table-striped')

    # 4. Generate Histograms & Convert to Base64
    plots_html = ""
    for col in numeric_df.columns:
        plt.figure(figsize=(6, 4))
        numeric_df[col].hist(bins=20, color='skyblue', edgecolor='black')
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.grid(axis='y', alpha=0.75)

        # Save plot to a memory buffer
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight')
        plt.close()
        
        # Encode buffer to base64 string
        data = base64.b64encode(buf.getbuffer()).decode("ascii")
        plots_html += f'<h3>{col}</h3><img src="data:image/png;base64,{data}"><br>'

    # 5. Build the Final HTML Report
    html_template = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>CSV Explorer Report: {os.path.basename(csv_path)}</title>
        <style>
            body {{ font-family: sans-serif; margin: 40px; line-height: 1.6; color: #333; }}
            .table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
            .table th, .table td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            .table th {{ background-color: #f2f2f2; }}
            h1, h2 {{ color: #2c3e50; }}
        </style>
    </head>
    <body>
        <h1>Automated Explorer Report</h1>
        <p><strong>Source File:</strong> {csv_path}</p>
        <hr>
        <h2>Summary Statistics</h2>
        {stats_table}
        <hr>
        <h2>Visual Distributions</h2>
        {plots_html}
    </body>
    </html>
    """

    # 6. Save the Report
    report_name = f"report_{os.path.splitext(os.path.basename(csv_path))[0]}.html"
    with open(report_name, "w") as f:
        f.write(html_template)
    
    print(f"Report successfully generated: {report_name}")

# if __name__ == "__main__":
if len(sys.argv) < 2:
    print("Usage: python explorer.py <path_to_csv>")
else:
    generate_report(sys.argv[1])


Error loading CSV: [Errno 2] No such file or directory: 'System_Anomalies.csv'
